# LR Search - Experiments A, B, C

Three independent Optuna studies, one per experiment config.  
**Only the learning rate is varied** - all other hyperparameters are frozen to the values in `configs/exp_{a,b,c}.toml`.

| Study | Config | Baseline LR | Architecture |
|-------|--------|-------------|------------|
| `lr_search_exp_a` | exp_a.toml | 3.6e-4 | d=288, L=8, H=4, SwiGLU, pre-norm |
| `lr_search_exp_b` | exp_b.toml | 4.2e-4 | d=448, L=10, H=8, SwiGLU, post-norm |
| `lr_search_exp_c` | exp_c.toml | 4.0e-4 | d=512, L=7, H=8, SwiGLU, pre-norm |

**Single-GPU execution** (no DDP). Effective batch is matched to the full DDP run:  
`batch_size=64 x grad_accum=8 x 1 GPU = 512 seqs/step` - identical to all three configs at 4/8 GPUs.

Search range: `5e-5 -> 3e-3` (log-uniform, 15 trials per experiment).  
For exact DDP matching use `scripts/optuna_lr_search.py` + `scripts/run_lr_search.sh`.

In [ ]:
import gc
import math
import random
import sys
import os
import warnings
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
from torch.utils.data import DataLoader

# -- Repo root resolution
_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "pyproject.toml").exists() and (p / "src").exists()),
    _cwd,
)
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tiny_llm.model.transformer import GPTModel
from tiny_llm.data.dataset import MemmapDataset
from tiny_llm.training.optimizer import create_optimizer, create_scheduler

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)

print(f"Repo root : {REPO_ROOT}")
print("Imports OK")


In [ ]:
# -- Device
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"

print(f"Device : {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
print(f"AMP    : {USE_AMP}")


In [ ]:
# -- Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -- Shared data paths
DATA_DIR  = REPO_ROOT / "data" / "tokenized"
TRAIN_BIN = DATA_DIR / "train.bin"
VAL_BIN   = DATA_DIR / "val.bin"
assert TRAIN_BIN.exists(), f"Missing: {TRAIN_BIN}"
assert VAL_BIN.exists(),   f"Missing: {VAL_BIN}"
print(f"train.bin : {TRAIN_BIN.stat().st_size / 1e9:.2f} GB")
print(f"val.bin   : {VAL_BIN.stat().st_size / 1e9:.2f} GB")

# -- Optuna DB (separate from the ablation DB)
DB_PATH = REPO_ROOT / "notebooks" / "tiny_llm_lr_search.db"
STORAGE = f"sqlite:///{DB_PATH}"
print(f"DB        : {DB_PATH}")


In [ ]:
# -- Experiment configs frozen from configs/exp_{a,b,c}.toml
# Only learning_rate will be varied by Optuna.

EXP_A = dict(
    vocab_size        = 50257,
    context_length    = 1024,
    n_layers          = 8,
    n_heads           = 4,
    d_model           = 288,
    d_ff              = 704,
    dropout           = 0.0,
    activation        = "swiglu",
    norm_position     = "pre",
    weight_tying      = False,
    bias              = False,
    qk_norm           = False,
    positional_encoding = "rope",
    rope_theta        = 10_000.0,
    weight_decay      = 0.28,
    betas             = (0.9, 0.95),
    baseline_lr       = 3.6e-4,
)

EXP_B = dict(
    vocab_size        = 50257,
    context_length    = 1024,
    n_layers          = 10,
    n_heads           = 8,
    d_model           = 448,
    d_ff              = 640,
    dropout           = 0.0,
    activation        = "swiglu",
    norm_position     = "post",
    weight_tying      = True,
    bias              = False,
    qk_norm           = False,
    positional_encoding = "rope",
    rope_theta        = 10_000.0,
    weight_decay      = 0.13,
    betas             = (0.9, 0.95),
    baseline_lr       = 4.2e-4,
)

EXP_C = dict(
    vocab_size        = 50257,
    context_length    = 1024,
    n_layers          = 7,
    n_heads           = 8,
    d_model           = 512,
    d_ff              = 640,
    dropout           = 0.0,
    activation        = "swiglu",
    norm_position     = "pre",
    weight_tying      = True,
    bias              = False,
    qk_norm           = False,
    positional_encoding = "rope",
    rope_theta        = 10_000.0,
    weight_decay      = 0.20,
    betas             = (0.9, 0.95),
    baseline_lr       = 4.0e-4,
)

for name, cfg in [("A", EXP_A), ("B", EXP_B), ("C", EXP_C)]:
    print(
        f"Exp {name}: d={cfg['d_model']} L={cfg['n_layers']} H={cfg['n_heads']} "
        f"d_ff={cfg['d_ff']} {cfg['activation']} {cfg['norm_position']}-norm  "
        f"wd={cfg['weight_decay']}  baseline_lr={cfg['baseline_lr']:.1e}"
    )


In [ ]:
# -- Training budget
#
# Single-GPU effective batch matches the full DDP runs:
#   configs A, B: 64 seqs x 2 accum x 4 GPUs = 512 seqs/step
#   config   C:   64 seqs x 1 accum x 8 GPUs = 512 seqs/step
# Single GPU: BATCH_SIZE=64 x GRAD_ACCUM=8 = 512 seqs/step

BATCH_SIZE   = 64
GRAD_ACCUM   = 8
EVAL_BATCH   = 256
TRAIN_STEPS  = 500   # per trial
WARMUP_STEPS = 100   # matches the configs
EVAL_STEPS   = 50    # val batches per evaluation
REPORT_EVERY = 100   # pruning check interval

EFF_TOK = BATCH_SIZE * GRAD_ACCUM * 1024
print(f"Effective batch : {BATCH_SIZE} x {GRAD_ACCUM} accum = {BATCH_SIZE*GRAD_ACCUM} seqs/step "
      f"-> {EFF_TOK/1e3:.0f}K tokens/step")
print(f"Train steps/trial: {TRAIN_STEPS}")

# -- Shared data loaders
train_ds = MemmapDataset(TRAIN_BIN, context_length=1024)
val_ds   = MemmapDataset(VAL_BIN,   context_length=1024)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=EVAL_BATCH, shuffle=False,
                          num_workers=4, pin_memory=True, drop_last=False)
print(f"train: {len(train_ds):,} samples  |  val: {len(val_ds):,} samples")


In [ ]:
# -- Core training utilities

def _next_batch(it, loader):
    try:
        return next(it)
    except StopIteration:
        return next(iter(loader))


@torch.no_grad()
def evaluate(model, n_steps):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_loader):
        if i >= n_steps:
            break
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast(device.type, enabled=USE_AMP):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def run_trial(trial, cfg, lr):
    """Train for TRAIN_STEPS steps and return final val loss."""
    model = None
    try:
        model_keys = [
            "vocab_size", "context_length", "n_layers", "n_heads", "d_model", "d_ff",
            "dropout", "activation", "norm_position", "weight_tying", "bias",
            "qk_norm", "positional_encoding", "rope_theta",
        ]
        model = GPTModel({k: cfg[k] for k in model_keys}).to(device)
        optimizer = create_optimizer(
            model, optimizer_name="adamw",
            learning_rate=lr,
            weight_decay=cfg["weight_decay"],
            betas=cfg["betas"],
        )
        scheduler = create_scheduler(
            optimizer, lr_schedule="cosine",
            warmup_steps=WARMUP_STEPS,
            max_steps=TRAIN_STEPS,
            min_lr_ratio=0.1,
        )
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
        model.train()
        train_iter = iter(train_loader)
        optimizer.zero_grad(set_to_none=True)

        for step in range(TRAIN_STEPS):
            for _ in range(GRAD_ACCUM):
                x, y = _next_batch(train_iter, train_loader)
                x, y = x.to(device), y.to(device)
                with torch.amp.autocast(device.type, enabled=USE_AMP):
                    _, loss = model(x, y)
                scaler.scale(loss / GRAD_ACCUM).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            if (step + 1) % REPORT_EVERY == 0:
                val_loss = evaluate(model, EVAL_STEPS)
                trial.report(val_loss, step)
                if trial.should_prune():
                    raise optuna.TrialPruned()

        return evaluate(model, EVAL_STEPS)

    except optuna.TrialPruned:
        raise
    except Exception as e:
        if any(kw in str(e) for kw in ("CUDA", "cuda", "out of memory", "AcceleratorError")):
            raise optuna.TrialPruned()
        raise
    finally:
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()


def show_results(study):
    """Print a sorted results table and highlight the best trial."""
    n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    print(f"Completed: {n_complete}  |  Pruned: {n_pruned}")
    if n_complete == 0:
        print("No completed trials.")
        return
    rows = []
    for t in study.trials:
        if t.state != optuna.trial.TrialState.COMPLETE:
            continue
        rows.append({
            "#":        t.number,
            "lr":       t.params["lr"],
            "val_loss": round(t.value, 4),
            "ppl":      round(math.exp(t.value), 1),
        })
    df = pd.DataFrame(rows).sort_values("val_loss").reset_index(drop=True)
    pd.set_option("display.float_format", "{:.4g}".format)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 120)
    print(df.to_string(index=False))
    best = study.best_trial
    print(f"\n{'='*52}")
    print(
        f"  BEST  trial #{best.number:>2}   "
        f"lr={best.params['lr']:.3e}   "
        f"loss={best.value:.4f}   PPL={math.exp(best.value):.1f}"
    )
    print(f"{'='*52}")


def best_callback(study, trial):
    """Print a one-line update after every completed trial."""
    if trial.state != optuna.trial.TrialState.COMPLETE:
        return
    best = study.best_trial
    marker = " << NEW BEST" if trial.number == best.number else ""
    print(
        f"  Trial {trial.number:>2}  lr={trial.params['lr']:.3e}  "
        f"val_loss={trial.value:.4f}  PPL={math.exp(trial.value):.1f}"
        f"  [best: trial {best.number}, lr={best.params['lr']:.3e}, "
        f"loss={best.value:.4f}]{marker}"
    )


print("Utilities defined.")


---
## Experiment A - LR Search

**Architecture:** d=288, L=8, H=4, d_ff=704, SwiGLU, pre-norm, no weight tying  
**Fixed params:** weight_decay=0.28, betas=(0.9, 0.95), cosine schedule  
**Baseline LR:** 3.6e-4  
**Search range:** 5e-5 -> 3e-3, 15 trials

In [ ]:
STUDY_A = "lr_search_exp_a"

def objective_a(trial):
    lr = trial.suggest_float("lr", 5e-5, 3e-3, log=True)
    return run_trial(trial, EXP_A, lr=lr)

study_a = optuna.create_study(
    direction="minimize",
    study_name=STUDY_A,
    storage=STORAGE,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=REPORT_EVERY),
)
n_remaining_a = max(0, 15 - sum(
    1 for t in study_a.trials if t.state == optuna.trial.TrialState.COMPLETE
))
print(f"Exp A - LR search: {n_remaining_a} trial(s) remaining")
study_a.optimize(objective_a, n_trials=n_remaining_a, callbacks=[best_callback],
                 show_progress_bar=False)
print("\n-- Exp A results --")
show_results(study_a)


---
## Experiment B - LR Search

**Architecture:** d=448, L=10, H=8, d_ff=640, SwiGLU, post-norm, weight tying  
**Fixed params:** weight_decay=0.13, betas=(0.9, 0.95), cosine schedule  
**Baseline LR:** 4.2e-4  
**Search range:** 5e-5 -> 3e-3, 15 trials

In [ ]:
STUDY_B = "lr_search_exp_b"

def objective_b(trial):
    lr = trial.suggest_float("lr", 5e-5, 3e-3, log=True)
    return run_trial(trial, EXP_B, lr=lr)

study_b = optuna.create_study(
    direction="minimize",
    study_name=STUDY_B,
    storage=STORAGE,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=REPORT_EVERY),
)
n_remaining_b = max(0, 15 - sum(
    1 for t in study_b.trials if t.state == optuna.trial.TrialState.COMPLETE
))
print(f"Exp B - LR search: {n_remaining_b} trial(s) remaining")
study_b.optimize(objective_b, n_trials=n_remaining_b, callbacks=[best_callback],
                 show_progress_bar=False)
print("\n-- Exp B results --")
show_results(study_b)


---
## Experiment C - LR Search

**Architecture:** d=512, L=7, H=8, d_ff=640, SwiGLU, pre-norm, weight tying  
**Fixed params:** weight_decay=0.20, betas=(0.9, 0.95), cosine schedule  
**Baseline LR:** 4.0e-4  
**Search range:** 5e-5 -> 3e-3, 15 trials

In [ ]:
STUDY_C = "lr_search_exp_c"

def objective_c(trial):
    lr = trial.suggest_float("lr", 5e-5, 3e-3, log=True)
    return run_trial(trial, EXP_C, lr=lr)

study_c = optuna.create_study(
    direction="minimize",
    study_name=STUDY_C,
    storage=STORAGE,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=REPORT_EVERY),
)
n_remaining_c = max(0, 15 - sum(
    1 for t in study_c.trials if t.state == optuna.trial.TrialState.COMPLETE
))
print(f"Exp C - LR search: {n_remaining_c} trial(s) remaining")
study_c.optimize(objective_c, n_trials=n_remaining_c, callbacks=[best_callback],
                 show_progress_bar=False)
print("\n-- Exp C results --")
show_results(study_c)


---
## Summary

Run this cell after all three searches complete.

In [ ]:
print("=" * 60)
print("LR SEARCH SUMMARY")
print("=" * 60)

for exp_name, study, cfg in [
    ("A", study_a, EXP_A),
    ("B", study_b, EXP_B),
    ("C", study_c, EXP_C),
]:
    n_done = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    if n_done == 0:
        print(f"Exp {exp_name}: no completed trials")
        continue
    best = study.best_trial
    improvement = (cfg["baseline_lr"] - best.params["lr"]) / cfg["baseline_lr"] * 100
    direction = "lower" if best.params["lr"] < cfg["baseline_lr"] else "higher"
    print(
        f"Exp {exp_name}: best_lr={best.params['lr']:.3e}  "
        f"(baseline={cfg['baseline_lr']:.1e}, {abs(improvement):.0f}% {direction})  "
        f"loss={best.value:.4f}  PPL={math.exp(best.value):.1f}  [{n_done} trials]"
    )

print("=" * 60)
print("Update learning_rate in the corresponding configs/exp_?.toml files.")


---
## DDP Results Viewer

After running `bash scripts/run_lr_search.sh`, use this cell to inspect
results from the DDP studies (study names: `lr_search_exp_{a,b,c}_ddp`).

In [ ]:
# Load results from the DDP Optuna studies.
# Run after scripts/run_lr_search.sh has completed.

DB_PATH_DDP = REPO_ROOT / "notebooks" / "tiny_llm_lr_search.db"
assert DB_PATH_DDP.exists(), f"DB not found: {DB_PATH_DDP} -- run scripts/run_lr_search.sh first"
STORAGE_DDP = f"sqlite:///{DB_PATH_DDP}"

DDP_META = {
    "a": {"baseline_lr": 3.6e-4, "label": "Exp A (d=288, L=8, pre-norm)"},
    "b": {"baseline_lr": 4.2e-4, "label": "Exp B (d=448, L=10, post-norm)"},
    "c": {"baseline_lr": 4.0e-4, "label": "Exp C (d=512, L=7, pre-norm)"},
}

for exp_key, meta in DDP_META.items():
    study_name = f"lr_search_exp_{exp_key}_ddp"
    try:
        s = optuna.load_study(study_name=study_name, storage=STORAGE_DDP)
    except Exception:
        print(f"  {study_name}: not found (not yet run)")
        continue
    label = meta["label"]
    print(f"\n{label}:")
    show_results(s)
